# Train MiniGPT on Shakespeare — your first real language model

In 04-transformer we built the architecture and generation loop.
Now we **train it on real text** and watch it learn to write English.

This is the complete loop: data → tokenize → train → generate.
Every LLM in the world does exactly this, just bigger.

## What you'll see
- Epoch 1: `asjdf kqwer zxcvb` (random garbage)
- Epoch 5: `the the the and and` (learns common words)
- Epoch 20: `what is this that thou dost speak` (learns structure)

## Setup
```bash
# On Colab, this dataset downloads automatically
# No extra packages needed — just PyTorch
```

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import urllib.request
import os

# --- Download Shakespeare text ---
DATA_URL = 'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt'
DATA_FILE = 'shakespeare.txt'

if not os.path.exists(DATA_FILE):
    urllib.request.urlretrieve(DATA_URL, DATA_FILE)
    print('Downloaded shakespeare.txt')

with open(DATA_FILE, 'r') as f:
    text = f.read()

print(f'Total characters: {len(text):,}')
print(f'First 200 chars:\n{text[:200]}')

## Character-level tokenizer

Real LLMs use BPE (byte-pair encoding) tokenizers like SentencePiece.
For learning: character-level is simpler and shows the same principles.

Each unique character gets an integer ID. That's it — that's a tokenizer.

In [ ]:
# Build character-level vocab
chars = sorted(set(text))
vocab_size = len(chars)

# Encoder: char → int,  Decoder: int → char
stoi = {ch: i for i, ch in enumerate(chars)}   # string to int
itos = {i: ch for i, ch in enumerate(chars)}   # int to string

encode = lambda s: [stoi[c] for c in s]
decode = lambda ids: ''.join([itos[i] for i in ids])

print(f'Vocab size: {vocab_size}')
print(f'Characters: {"".join(chars[:30])}...')
print(f'\nEncode "hello": {encode("hello")}')
print(f'Decode back:    {decode(encode("hello"))}')

In [ ]:
# Encode entire text and split into train/val
data = torch.tensor(encode(text), dtype=torch.long)
print(f'Encoded tensor shape: {data.shape}')

split = int(0.9 * len(data))
train_data = data[:split]
val_data   = data[split:]
print(f'Train: {len(train_data):,} tokens')
print(f'Val:   {len(val_data):,} tokens')

In [ ]:
# --- Batch generator ---
# Creates random chunks of text for training
# x = input tokens,  y = target tokens (shifted by 1)
#
# Example: "hello world"
#   x = [h, e, l, l, o,  , w, o, r, l]
#   y = [e, l, l, o,  , w, o, r, l, d]   ← predict next char at each position

BLOCK_SIZE = 128   # context length (how many tokens model sees at once)
BATCH_SIZE = 32

def get_batch(split):
    data_split = train_data if split == 'train' else val_data
    # Random starting positions
    ix = torch.randint(len(data_split) - BLOCK_SIZE, (BATCH_SIZE,))
    x  = torch.stack([data_split[i   : i+BLOCK_SIZE]     for i in ix])
    y  = torch.stack([data_split[i+1 : i+BLOCK_SIZE+1]   for i in ix])
    return x, y

x, y = get_batch('train')
print(f'Input shape:  {x.shape}')    # [32, 128]
print(f'Target shape: {y.shape}')    # [32, 128]
print(f'\nInput[0][:20]:  {decode(x[0][:20].tolist())}')
print(f'Target[0][:20]: {decode(y[0][:20].tolist())}')

## Model — MiniGPT with production tricks

Same architecture from 04-transformer, but now with two tricks every real GPT uses:

**1. Weight tying** — the input embedding and output head share the same weight matrix.
Think about it: the embedding maps `token_id → vector`. The output head maps `vector → logits over tokens`.
These are inverse operations — sharing weights forces them to learn a consistent representation.
GPT-2, GPT-3, LLaMA all do this. Saves parameters AND improves quality.

**2. Proper initialization** — from Level 3 backprop notebook, we learned that bad init → vanishing/exploding gradients.
GPT-2 scales residual layer outputs by `1/sqrt(num_layers)` to keep activations stable as the network gets deeper.

In [ ]:
# --- Reuse transformer components from 04-transformer ---

def scaled_dot_product_attention(Q, K, V, mask=None):
    d_k    = Q.shape[-1]
    scores = Q @ K.transpose(-2, -1) / math.sqrt(d_k)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))
    weights = F.softmax(scores, dim=-1)
    return weights @ V, weights


class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        self.d_model   = d_model
        self.num_heads = num_heads
        self.d_k       = d_model // num_heads
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

    def forward(self, x, mask=None):
        b, s, _ = x.shape
        Q = self.W_q(x).reshape(b, s, self.num_heads, self.d_k).transpose(1, 2)
        K = self.W_k(x).reshape(b, s, self.num_heads, self.d_k).transpose(1, 2)
        V = self.W_v(x).reshape(b, s, self.num_heads, self.d_k).transpose(1, 2)
        context, _ = scaled_dot_product_attention(Q, K, V, mask)
        return self.W_o(context.transpose(1, 2).reshape(b, s, self.d_model))


class TransformerBlock(nn.Module):
    def __init__(self, d_model, num_heads, ffn_dim, dropout=0.1):
        super().__init__()
        self.attn    = MultiHeadAttention(d_model, num_heads)
        self.ffn     = nn.Sequential(nn.Linear(d_model, ffn_dim), nn.GELU(), nn.Linear(ffn_dim, d_model))
        self.norm1   = nn.LayerNorm(d_model)
        self.norm2   = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        x = x + self.dropout(self.attn(self.norm1(x), mask))
        x = x + self.dropout(self.ffn(self.norm2(x)))
        return x


class MiniGPT(nn.Module):
    def __init__(self, vocab_size, d_model, num_heads, num_layers, max_seq_len, ffn_dim):
        super().__init__()
        self.token_embed = nn.Embedding(vocab_size, d_model)
        self.pos_embed   = nn.Embedding(max_seq_len, d_model)
        self.blocks      = nn.ModuleList([TransformerBlock(d_model, num_heads, ffn_dim) for _ in range(num_layers)])
        self.norm        = nn.LayerNorm(d_model)
        self.head        = nn.Linear(d_model, vocab_size, bias=False)

        # --- WEIGHT TYING ---
        # Input embedding and output head share the same weight matrix
        # Embedding: [vocab_size, d_model]  Output head: [d_model, vocab_size] (transposed)
        # This is NOT a copy — they are literally the same tensor in memory
        self.head.weight = self.token_embed.weight    # ← the key line

        # --- PROPER INITIALIZATION (GPT-2 style) ---
        self._init_weights(num_layers)

    def _init_weights(self, num_layers):
        """GPT-2 initialization: normal(0, 0.02) for most weights,
        scale residual outputs by 1/sqrt(num_layers) to keep deep networks stable."""
        for name, p in self.named_parameters():
            if p.dim() >= 2:
                nn.init.normal_(p, mean=0.0, std=0.02)
            elif 'bias' in name:
                nn.init.zeros_(p)

        # Scale residual projections — prevents activations from growing with depth
        # This is the trick from GPT-2 paper
        for block in self.blocks:
            nn.init.normal_(block.attn.W_o.weight, mean=0.0, std=0.02 / math.sqrt(2 * num_layers))
            nn.init.normal_(block.ffn[2].weight,   mean=0.0, std=0.02 / math.sqrt(2 * num_layers))

    def forward(self, token_ids):
        b, seq_len = token_ids.shape
        x    = self.token_embed(token_ids) + self.pos_embed(torch.arange(seq_len, device=token_ids.device))
        mask = torch.tril(torch.ones(seq_len, seq_len, device=token_ids.device)).unsqueeze(0).unsqueeze(0)
        for block in self.blocks:
            x = block(x, mask)
        return self.head(self.norm(x))


# --- Hyperparameters ---
device = 'cuda' if torch.cuda.is_available() else 'cpu'

model = MiniGPT(
    vocab_size   = vocab_size,
    d_model      = 256,
    num_heads    = 4,
    num_layers   = 6,
    max_seq_len  = BLOCK_SIZE,
    ffn_dim      = 512,
).to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f'MiniGPT: {total_params:,} parameters')
print(f'Device:  {device}')

# Verify weight tying — same memory address
print(f'\nWeight tying active: {model.head.weight.data_ptr() == model.token_embed.weight.data_ptr()}')

## Training loop — with all the production tricks

Same 5-step loop from Level 3, plus everything you learned:
- **AdamW** optimizer (Level 3, `03-optimizer`)
- **Gradient clipping** (Level 3, `04-training-loop`) — max_norm=1.0
- **LR warmup + cosine decay** (Level 3, `03-optimizer`) — gradual start, smooth finish

**Random baseline:** `-log(1/vocab_size)` — if the model predicts uniformly, loss = `log(65) ~ 4.17`.
If loss drops below this, the model is learning.

In [ ]:
# --- LR schedule (from Level 3 optimizer notebook) ---
def get_lr(step, warmup_steps, max_steps, max_lr, min_lr):
    if step < warmup_steps:
        return max_lr * step / warmup_steps
    elif step >= max_steps:
        return min_lr
    progress = (step - warmup_steps) / (max_steps - warmup_steps)
    return min_lr + 0.5 * (max_lr - min_lr) * (1 + math.cos(math.pi * progress))


# --- Hyperparameters ---
MAX_STEPS    = 3000     # increase for better results (5000-10000 on GPU)
MAX_LR       = 3e-4
MIN_LR       = 3e-5
WARMUP_STEPS = 200

optimizer = torch.optim.AdamW(model.parameters(), lr=MAX_LR, weight_decay=0.01)

random_loss = -math.log(1 / vocab_size)
print(f'Random baseline loss: {random_loss:.2f} — model must beat this\n')

# --- Training ---
for step in range(MAX_STEPS):
    # Set learning rate for this step (warmup + cosine decay)
    lr = get_lr(step, WARMUP_STEPS, MAX_STEPS, MAX_LR, MIN_LR)
    for param_group in optimizer.param_groups:
        param_group['lr'] = lr

    # 1. Get batch
    x, y = get_batch('train')
    x, y = x.to(device), y.to(device)

    # 2. Forward
    logits = model(x)
    loss   = F.cross_entropy(logits.view(-1, vocab_size), y.view(-1))

    # 3. Backward
    optimizer.zero_grad()
    loss.backward()

    # 4. Gradient clipping (safety — from Level 3 training loop)
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

    # 5. Update
    optimizer.step()

    if step % 500 == 0 or step == MAX_STEPS - 1:
        print(f'Step {step:5d} | Loss: {loss.item():.4f} | LR: {lr:.6f}')

print(f'\nFinal loss: {loss.item():.4f} (random would be {random_loss:.2f})')

## Generate text — the payoff

Now use the autoregressive generation loop with sampling from 04-transformer.

In [ ]:
def top_p_sampling(logits, p=0.9):
    sorted_logits, sorted_indices = logits.sort(descending=True)
    cumulative_probs = torch.softmax(sorted_logits, dim=-1).cumsum(dim=-1)
    mask = cumulative_probs - torch.softmax(sorted_logits, dim=-1) >= p
    sorted_logits[mask] = float('-inf')
    return sorted_logits.gather(1, sorted_indices.argsort(1))


@torch.no_grad()
def generate(model, prompt, max_new_tokens=200, temperature=0.8, top_p=0.9):
    model.eval()
    tokens = torch.tensor([encode(prompt)], device=device)

    for _ in range(max_new_tokens):
        # Crop to block size (model can't handle longer than it was built for)
        tokens_cropped = tokens[:, -BLOCK_SIZE:]
        logits         = model(tokens_cropped)
        next_logits    = logits[:, -1, :] / temperature
        next_logits    = top_p_sampling(next_logits, p=top_p)
        probs          = torch.softmax(next_logits, dim=-1)
        next_token     = torch.multinomial(probs, num_samples=1)
        tokens         = torch.cat([tokens, next_token], dim=1)

    model.train()
    return decode(tokens[0].tolist())


# Generate with different prompts
print('=' * 60)
print('PROMPT: "The king"')
print('=' * 60)
print(generate(model, 'The king', max_new_tokens=200))

print('\n' + '=' * 60)
print('PROMPT: "To be or not"')
print('=' * 60)
print(generate(model, 'To be or not', max_new_tokens=200))

## Validation loss + perplexity

How good is the model? Check on text it's never seen.

In [ ]:
@torch.no_grad()
def evaluate(model, num_batches=50):
    model.eval()
    total_loss = 0.0
    for _ in range(num_batches):
        x, y = get_batch('val')
        x, y = x.to(device), y.to(device)
        logits = model(x)
        loss   = F.cross_entropy(logits.view(-1, vocab_size), y.view(-1))
        total_loss += loss.item()
    avg_loss = total_loss / num_batches
    perplexity = math.exp(avg_loss)
    model.train()
    return avg_loss, perplexity

val_loss, ppl = evaluate(model)
print(f'Validation loss: {val_loss:.4f}')
print(f'Perplexity:      {ppl:.2f}')
print(f'\n(Lower = better. Random baseline perplexity = {vocab_size})')

## What you just built

```
Raw text  →  character tokenizer  →  tensor batches
    ↓
MiniGPT (transformer, causal mask, 6 layers)
    ↓
Training loop (AdamW, cross-entropy loss)
    ↓
Autoregressive generation (top-p sampling)
    ↓
Shakespeare-like text output
```

**This is identical to how GPT-2 was trained.** The only differences:
- GPT-2: BPE tokenizer, 1.5B params, trained on 40GB of internet text
- Yours: character tokenizer, ~3M params, trained on 1MB of Shakespeare

Same architecture. Same training loop. Same generation method.

**Next:** Level 5 (HuggingFace, fine-tuning) builds on top of pretrained models.
Level 6 (Flash Attention, KV Cache) optimizes the parts you just built.